In [1]:
import numpy as np 
import pandas as pd 
from pathlib import Path

In [2]:
CLEANED_DATAFRAME_LOCATION = Path('..') / 'datasets' / 'cleaned_dataset.csv'
cleaned_df = pd.read_csv(CLEANED_DATAFRAME_LOCATION)

# ═════════════════════════════════════════════════════════════════════════════
# BUILD PRODUCT-LEVEL COST STRUCTURE TABLE
# ═════════════════════════════════════════════════════════════════════════════
product_df = (
    cleaned_df.groupby('Product Name').agg(
        Total_Sales=('Sales', 'sum'),
        Total_Cost=('Cost', 'sum'),
        Total_Profit=('Gross Profit', 'sum'),
        Total_Units=('Units', 'sum'),
        Num_Orders=('Order ID', 'nunique'),
    ).sort_values('Total_Sales', ascending=False)
)

# Derived metrics
product_df['Margin (%)'] = product_df['Total_Profit'] / product_df['Total_Sales'] * 100
product_df['Cost Ratio (%)'] = product_df['Total_Cost'] / product_df['Total_Sales'] * 100
product_df['Avg Price/Unit'] = product_df['Total_Sales'] / product_df['Total_Units']
product_df['Avg Cost/Unit'] = product_df['Total_Cost'] / product_df['Total_Units']
product_df['Markup (%)'] = (product_df['Avg Price/Unit'] - product_df['Avg Cost/Unit']) / product_df['Avg Cost/Unit'] * 100
product_df['Profit/Unit'] = product_df['Total_Profit'] / product_df['Total_Units']
product_df['Revenue Share (%)'] = product_df['Total_Sales'] / product_df['Total_Sales'].sum() * 100
product_df['Profit Share (%)'] = product_df['Total_Profit'] / product_df['Total_Profit'].sum() * 100

# Overall benchmarks
overall_margin = product_df['Total_Profit'].sum() / product_df['Total_Sales'].sum() * 100
overall_cost_ratio = product_df['Total_Cost'].sum() / product_df['Total_Sales'].sum() * 100
median_margin = product_df['Margin (%)'].median()
median_markup = product_df['Markup (%)'].median()

In [3]:
# ═════════════════════════════════════════════════════════════════════════════
# SECTION 1: Cost vs Sales Scatter Analysis
# ═════════════════════════════════════════════════════════════════════════════
print("=" * 85)
print("SECTION 1: COST vs SALES SCATTER ANALYSIS")
print("=" * 85)
print(f"\n  Overall Average Margin: {overall_margin:.1f}%")
print(f"  Overall Cost Ratio:    {overall_cost_ratio:.1f}%")
print(f"  Median Product Margin: {median_margin:.1f}%")
print(f"  Median Product Markup: {median_markup:.1f}%\n")

print("--- Product Cost Structure Table ---\n")
display_cols = ['Total_Sales', 'Total_Cost', 'Total_Profit', 'Margin (%)',
                'Cost Ratio (%)', 'Avg Price/Unit', 'Avg Cost/Unit', 'Markup (%)']
print(product_df[display_cols].round(2).to_string())

SECTION 1: COST vs SALES SCATTER ANALYSIS

  Overall Average Margin: 65.9%
  Overall Cost Ratio:    34.1%
  Median Product Margin: 62.3%
  Median Product Markup: 165.3%

--- Product Cost Structure Table ---

                                   Total_Sales  Total_Cost  Total_Profit  Margin (%)  Cost Ratio (%)  Avg Price/Unit  Avg Cost/Unit  Markup (%)
Product Name                                                                                                                                   
Wonka Bar - Triple Dazzle Caramel     28485.00     9874.80      18610.20       65.33           34.67            3.75           1.30      188.46
Wonka Bar -Scrumdiddlyumptious        27874.80     8517.30      19357.50       69.44           30.56            3.60           1.10      227.27
Wonka Bar - Milk Chocolate            26867.75     9424.38      17443.37       64.92           35.08            3.25           1.14      185.09
Wonka Bar - Fudge Mallows             24890.40     8296.80      16593.60

In [4]:
# ═════════════════════════════════════════════════════════════════════════════
# SECTION 2: Identify Cost-Heavy, Margin-Poor Products & Pricing Issues
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 85)
print("SECTION 2: COST-HEAVY, MARGIN-POOR PRODUCTS & PRICING INEFFICIENCIES")
print("=" * 85)

# ── 2a. Cost-Heavy Products (Cost Ratio > overall average) ──────────────────
cost_heavy = product_df[product_df['Cost Ratio (%)'] > overall_cost_ratio].sort_values(
    'Cost Ratio (%)', ascending=False
)

print(f"\n--- 2a. Cost-Heavy Products (Cost Ratio > {overall_cost_ratio:.1f}% avg) ---\n")
if cost_heavy.empty:
    print("  None found.")
else:
    for name, row in cost_heavy.iterrows():
        severity = "CRITICAL" if row['Cost Ratio (%)'] > 80 else "HIGH" if row['Cost Ratio (%)'] > 60 else "MODERATE"
        print(f"  >> {name}")
        print(f"     Cost Ratio: {row['Cost Ratio (%)']:.1f}%  |  "
              f"Margin: {row['Margin (%)']:.1f}%  |  "
              f"Markup: {row['Markup (%)']:.0f}%  |  Severity: {severity}")

# ── 2b. Margin-Poor Products (Margin < overall average) ─────────────────────
margin_poor = product_df[product_df['Margin (%)'] < overall_margin].sort_values(
    'Margin (%)', ascending=True
)

print(f"\n--- 2b. Margin-Poor Products (Margin < {overall_margin:.1f}% avg) ---\n")
if margin_poor.empty:
    print("  None found.")
else:
    for name, row in margin_poor.iterrows():
        print(f"  >> {name}")
        print(f"     Margin: {row['Margin (%)']:.1f}%  |  "
              f"Revenue Share: {row['Revenue Share (%)']:.1f}%  |  "
              f"Profit Share: {row['Profit Share (%)']:.1f}%")

# ── 2c. Pricing Inefficiencies ──────────────────────────────────────────────
print(f"\n--- 2c. Pricing Inefficiencies ---\n")
print(f"  Products where Profit Share significantly trails Revenue Share:\n")

product_df['Profit-Revenue Gap'] = product_df['Profit Share (%)'] - product_df['Revenue Share (%)']
pricing_issues = product_df[product_df['Profit-Revenue Gap'] < -0.5].sort_values('Profit-Revenue Gap')

for name, row in pricing_issues.iterrows():
    print(f"  >> {name}")
    print(f"     Revenue Share: {row['Revenue Share (%)']:.2f}%  |  "
          f"Profit Share: {row['Profit Share (%)']:.2f}%  |  "
          f"Gap: {row['Profit-Revenue Gap']:+.2f}%")


SECTION 2: COST-HEAVY, MARGIN-POOR PRODUCTS & PRICING INEFFICIENCIES

--- 2a. Cost-Heavy Products (Cost Ratio > 34.1% avg) ---

  >> Kazookles
     Cost Ratio: 92.3%  |  Margin: 7.7%  |  Markup: 8%  |  Severity: CRITICAL
  >> Fun Dip
     Cost Ratio: 60.0%  |  Margin: 40.0%  |  Markup: 67%  |  Severity: MODERATE
  >> SweeTARTS
     Cost Ratio: 53.3%  |  Margin: 46.7%  |  Markup: 88%  |  Severity: MODERATE
  >> Nerds
     Cost Ratio: 53.3%  |  Margin: 46.7%  |  Markup: 87%  |  Severity: MODERATE
  >> Lickable Wallpaper
     Cost Ratio: 50.0%  |  Margin: 50.0%  |  Markup: 100%  |  Severity: MODERATE
  >> Wonka Gum
     Cost Ratio: 48.0%  |  Margin: 52.0%  |  Markup: 108%  |  Severity: MODERATE
  >> Fizzy Lifting Drinks
     Cost Ratio: 40.0%  |  Margin: 60.0%  |  Markup: 150%  |  Severity: MODERATE
  >> Laffy Taffy
     Cost Ratio: 37.7%  |  Margin: 62.3%  |  Markup: 165%  |  Severity: MODERATE
  >> Wonka Bar - Milk Chocolate
     Cost Ratio: 35.1%  |  Margin: 64.9%  |  Markup: 185%  | 

In [5]:
# ═════════════════════════════════════════════════════════════════════════════
# SECTION 3: Action Flags — Repricing, Cost Renegotiation, Discontinuation
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 85)
print("SECTION 3: PRODUCT ACTION FLAGS")
print("=" * 85)

def flag_product(row):
    """Determine recommended action for each product."""
    flags = []

    margin = row['Margin (%)']
    cost_ratio = row['Cost Ratio (%)']
    markup = row['Markup (%)']
    rev_share = row['Revenue Share (%)']
    profit_gap = row['Profit-Revenue Gap']

    # DISCONTINUATION: Very low margin + very low revenue contribution
    if margin < 15 and rev_share < 1:
        flags.append('DISCONTINUATION REVIEW')

    # COST RENEGOTIATION: High cost ratio but product has decent sales volume
    if cost_ratio > 50 and margin < overall_margin:
        flags.append('COST RENEGOTIATION')

    # REPRICING: Margin below avg but product sells well, or large profit-revenue gap
    needs_repricing = False
    if margin < overall_margin and rev_share > 0.5:
        needs_repricing = True
    if profit_gap < -0.5:
        needs_repricing = True
    if markup < median_markup and margin < overall_margin:
        needs_repricing = True
    if needs_repricing:
        flags.append('REPRICING')

    if not flags:
        flags.append('NO ACTION NEEDED')

    return flags


product_df['Action Flags'] = product_df.apply(flag_product, axis=1)
product_df['Primary Action'] = product_df['Action Flags'].apply(lambda x: x[0])

# Print grouped by action
print()
for action in ['DISCONTINUATION REVIEW', 'COST RENEGOTIATION', 'REPRICING', 'NO ACTION NEEDED']:
    flagged = product_df[product_df['Action Flags'].apply(lambda x: action in x)]
    if flagged.empty:
        continue

    icon = {'DISCONTINUATION REVIEW': '🔴', 'COST RENEGOTIATION': '🟠',
            'REPRICING': '🟡', 'NO ACTION NEEDED': '✅'}.get(action, '⬜')

    print(f"\n  {icon} {action} ({len(flagged)} products)")
    print(f"  {'─' * 70}")

    for name, row in flagged.iterrows():
        all_flags = ', '.join(row['Action Flags'])
        print(f"    >> {name}")
        print(f"       Margin: {row['Margin (%)']:.1f}%  |  "
              f"Cost Ratio: {row['Cost Ratio (%)']:.1f}%  |  "
              f"Markup: {row['Markup (%)']:.0f}%  |  "
              f"Rev Share: {row['Revenue Share (%)']:.2f}%")
        if len(row['Action Flags']) > 1:
            print(f"       All flags: {all_flags}")

        # Specific reasoning
        reasons = []
        if row['Margin (%)'] < 15:
            reasons.append(f"Critically low margin ({row['Margin (%)']:.1f}%)")
        if row['Cost Ratio (%)'] > 50:
            reasons.append(f"Cost consumes {row['Cost Ratio (%)']:.0f}% of revenue")
        if row['Markup (%)'] < 30:
            reasons.append(f"Insufficient markup ({row['Markup (%)']:.0f}%)")
        if row['Profit-Revenue Gap'] < -0.5:
            reasons.append(f"Profit share trails revenue by {abs(row['Profit-Revenue Gap']):.1f}pp")
        if reasons:
            print(f"       Reason:  {'; '.join(reasons)}")
        print()


SECTION 3: PRODUCT ACTION FLAGS


  🔴 DISCONTINUATION REVIEW (1 products)
  ──────────────────────────────────────────────────────────────────────
    >> Kazookles
       Margin: 7.7%  |  Cost Ratio: 92.3%  |  Markup: 8%  |  Rev Share: 0.85%
       All flags: DISCONTINUATION REVIEW, COST RENEGOTIATION, REPRICING
       Reason:  Critically low margin (7.7%); Cost consumes 92% of revenue; Insufficient markup (8%); Profit share trails revenue by 0.8pp


  🟠 COST RENEGOTIATION (4 products)
  ──────────────────────────────────────────────────────────────────────
    >> Kazookles
       Margin: 7.7%  |  Cost Ratio: 92.3%  |  Markup: 8%  |  Rev Share: 0.85%
       All flags: DISCONTINUATION REVIEW, COST RENEGOTIATION, REPRICING
       Reason:  Critically low margin (7.7%); Cost consumes 92% of revenue; Insufficient markup (8%); Profit share trails revenue by 0.8pp

    >> SweeTARTS
       Margin: 46.7%  |  Cost Ratio: 53.3%  |  Markup: 88%  |  Rev Share: 0.04%
       All flags: COST RENEGOTI

In [6]:
# ═════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 85)
print("FINAL COST STRUCTURE SUMMARY")
print("=" * 85)

action_counts = {}
for _, row in product_df.iterrows():
    for flag in row['Action Flags']:
        action_counts[flag] = action_counts.get(flag, 0) + 1

print(f"\n  Products by Action Flag:")
for action, count in sorted(action_counts.items()):
    print(f"    {action}: {count} products")

print(f"\n  Key Findings:")
worst = product_df.loc[product_df['Margin (%)'].idxmin()]
best = product_df.loc[product_df['Margin (%)'].idxmax()]
print(f"    Lowest margin:  {worst.name} ({worst['Margin (%)']:.1f}%)")
print(f"    Highest margin: {best.name} ({best['Margin (%)']:.1f}%)")
print(f"    Margin spread:  {best['Margin (%)'] - worst['Margin (%)']:.1f} percentage points")


FINAL COST STRUCTURE SUMMARY

  Products by Action Flag:
    COST RENEGOTIATION: 4 products
    DISCONTINUATION REVIEW: 1 products
    NO ACTION NEEDED: 6 products
    REPRICING: 9 products

  Key Findings:
    Lowest margin:  Kazookles (7.7%)
    Highest margin: Everlasting Gobstopper (80.0%)
    Margin spread:  72.3 percentage points
